In [1]:
import kagglehub
dataset_of_pdf_files_path = kagglehub.dataset_download('manisha717/dataset-of-pdf-files')
print(f"The dataset is loaded to the path: {dataset_of_pdf_files_path}")

/Users/hglabplhak/Documents/Projects/DLMDSDL01/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The dataset is loaded to the path: /Users/hglabplhak/.cache/kagglehub/datasets/manisha717/dataset-of-pdf-files/versions/1


In [5]:
from AskByRAG import get_dbpath, set_api_env_and_keys
from pypdf import PdfReader
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
#from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
import os


NameError: name 'nn' is not defined

In [ ]:
def build_vectors(complete_content):
     # 2. Embed and Store in Vector DB (Chroma)
    splitter = CharacterTextSplitter(
                chunk_size = 2000,
                chunk_overlap = 180,
                )

    chunks = splitter.create_documents(complete_content)

    # 2. Embed and Store in Vector DB (Chroma)
    embeddings = OpenAIEmbeddings()
    vector_db = Chroma.from_documents(chunks, embedding=embeddings, persist_directory=get_dbpath())
    return vector_db

In [ ]:
def extract_text_from_pdf(file_path):
    # creating a pdf reader object
    reader = PdfReader(file_path)
    content = ''
# printing number of pages in pdf file
    page_count = len(reader.pages)
    print(f'Number of pages: {page_count}')
    # getting a specific page from the pdf file
    for index in range(page_count):
        page = reader.pages[index]
        text = page.extract_text()
        content += text
    return content

In [ ]:
def read_all_docs(data_paths):
    #absolute_path = data_path + '/Pdf'
    docs_string = ''
    for data_path in data_paths:
        filenames = os.listdir(data_path)
        for filename in filenames:
            file_path = os.path.join(data_path, filename)
            print(f"Processing {file_path}")
            try:
                text = extract_text_from_pdf(file_path)
                docs_string += text
            except Exception as e:
                print(f"Error on {filename}: {e}")
                docs_string += "Failed"
                return -1, docs_string
    return 0, [docs_string]


In [ ]:
#docs_array = read_all_docs(dataset_of_pdf_files_path)
set_api_env_and_keys()
absolute_path = dataset_of_pdf_files_path + '/Pdf'
ret_code, complete_content = read_all_docs(['/Users/hglabplhak/pdfdb', '/Users/hglabplhak/pdfprivdb', absolute_path])
if ret_code == 0:
    print(complete_content[0])
    print(f"Count of docs: {len(complete_content)}")
    print("build vector")
    vector_db = build_vectors(complete_content)
    print('ready')